# LAB 2: LINEAR REGRESSION AND OPTIMIZATION
## NAME: ENYA TRAN

In [55]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [56]:
!pip install ucimlrepo -q

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import plotly.express as px

seoul = fetch_ucirepo(id=560)

bike = seoul.data.original.copy()

bike.head()



,Date,Rented Bike Count,Hour,Temperature,Humidity,Wind speed,Visibility,Dew point temperature,Solar Radiation,Rainfall,Snowfall,Seasons,Holiday,Functioning Day
0,1/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,1/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,1/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,1/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,1/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [57]:
bike.columns

Index(['Date', 'Rented Bike Count', 'Hour', 'Temperature', 'Humidity',
       'Wind speed', 'Visibility', 'Dew point temperature', 'Solar Radiation',
       'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning Day'],
      dtype='str')

In [58]:
bike = bike[bike['Functioning Day'] == 'Yes'].copy()
bike

,Date,Rented Bike Count,Hour,Temperature,Humidity,Wind speed,Visibility,Dew point temperature,Solar Radiation,Rainfall,Snowfall,Seasons,Holiday,Functioning Day
0,1/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,1/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,1/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,1/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,1/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,30/11/2018,1003,19,4.2,34,2.6,1894,-10.3,0.0,0.0,0.0,Autumn,No Holiday,Yes
8756,30/11/2018,764,20,3.4,37,2.3,2000,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8757,30/11/2018,694,21,2.6,39,0.3,1968,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8758,30/11/2018,712,22,2.1,41,1.0,1859,-9.8,0.0,0.0,0.0,Autumn,No Holiday,Yes


## Part 1: Explore the Relationship

1. Create a scatterplot of Rented Bike Count versus Temperature

In [59]:

fig = px.scatter(
    bike,
    x="Temperature",
    y="Rented Bike Count",
    title='Relationship between Temperature and Rented Bike Count'
)
fig.update_traces(textposition = "top center")
fig.show()

2. Describe the relationship. Comment briefly on its direction, approximate form, and variability
- The temperature and the rented bike count have a positive relationship. Higher/warmer temperature will have more rentals and vice versa. I think that the pattern is not perfectly linear because there are bike declines at the highest temperatures and there is great variability at a given temperature

3. Does a straight-line model appear reasonable as a first model?
- Yes, I think a straight-line model is reasonable because it can capture the overall positive association between bike rental and temperature. As discussed in lecture, linear regression is also a simple foundation for machine learning that is easy to interpret and useful as a baseline. However, it will not capture the big variation at a given temperature and the curvature at high temperatures

## Part 2: Fit and Interpret the Model

1. Fit the model and report bo^ and b1^

In [60]:
bike_model = LinearRegression()

bike_model.fit(
    bike[["Temperature"]],
    bike["Rented Bike Count"]
)

ols_b0 = bike_model.intercept_
ols_b1 = bike_model.coef_[0]

print("Intercept:", ols_b0)
print("Slope:", ols_b1)

Intercept: 347.7708421134778
Slope: 29.863318947981266


2. Interpret b1^: A 1C increase in temperature is associated with ~30 more bike rentals during an hour. The slope unit is an hourly bike rental per celcius degree

3. Interpret b0^: At 0C, the model predicts there are ~348 bike rentals during an hour. This intercept is meaningful because 0C is within the observed temperature range.

4. Suppose two hours differ in temperature by 10C:
- Delta(y) = b1^ * delta(x) = 29.86 * 10 = 298.63
- A 10C increase in temperatre is predicted to have ~299 more bike rentals

## Part 3: OLS as Optimization

1. Explain what a residual represents. What does a positive residual mean? What does a negative residual mean?

- A residual, $e_i = y_i - \hat{y}_i$, is the difference between the observed bike rentals versus the predicted bike rentals. A positive residual means the model underpredicted rentals, while a negative number means the model overpredicted rentals.

2. Why do we square the residuals before adding them? Give two reasons
- Squaring the residuals will prevent positive and negative outputs from canceling each other out
- Squaring will give larger errors (variability) greater penalty. This helps make the model more sensitive to large prediction errors.

3. Suppose Model A has SSE = 8,500,000 and Model B has SSE = 6,200,000. Which model does OLS prefer? Why
- OLS will prefer Model B because it has a lower SSE at 6,200,000. This means Model B has less total sum of squared errors, which is the objective OLS minimizes.

4. Does a smaller training SSE necessarily mean better prediction on new data? Explain.
- No. A smaller training SSE means the model makes less sum of squared errors on the training data. However, it may have fit noise or learned random patterns that will not hold for new observations. This is called overfitting. Therefore, we also have to check its performance on new data to see how good its prediction is.

## Part 4: Gradient Descent

1. In your own words, what do the two partial derivaties tell us? Why do we move in the opposite direction?
- The two partial derivatives tell us how the MSE changes when we adjust $b_0$ or $b_1$, holding the other coefficient fixed. They both then form the gradient which is the steepest increase in loss. So, we have to move in the opposite direction because our goal is to reduce loss to the minimum

2. Implement gradient descent to estimate b0 and b1. Record the MSE at each iteration

In [61]:
x = bike["Temperature"]
y = bike["Rented Bike Count"]
n = len(y)

In [62]:
learning_rate = 1e-8

In [63]:
b0 = 0.0
b1 = 0.0

y_hat = b0 + b1*x
residual = y - y_hat

iterations = 10000
checkpoints = [0,1,5,10,25,50,100,250,500,1000,2500,5000,7500,100000]

history = {
    0: {
        "residual": np.mean(residual **2),
        "b0": b0,
        "b1": b1
    }
}

for i in range(1, iterations + 1):
    y_hat = b0 + b1*x
    residual = y - y_hat
    grad_b0 = -(2/n) * np.sum(residual)
    grad_b1 = -(2/n) * np.sum(residual * x)
    b0 = b0 - learning_rate * grad_b0
    b1 = b1 - learning_rate * grad_b1

    if i in checkpoints:
        history[i] = {
            "residual": np.mean(residual**2),
            "b0": b0,
            "b1": b1
        }

3. Report your final estimates of b0 and b1. How close are they to the OLS estimates from Part 2?
- The gradient descent's esimates are much further from those of OLS. For gradient descent, $b_0\approx 0.1424$ and $b_1\approx 2.6542$ as compared to OLS with $b_0\approx 347.77$ and $b_1\approx 29.86$ 

In [64]:
grad_b0 = b0
grad_b1 = b1
print('OLS')
print('Intercept:', ols_b0)
print('Slope:   ', ols_b1)

print()

print('Gradient Descent')
print('Intercept:', grad_b0)
print('Slope:   ', grad_b1)

OLS
Intercept: 347.7708421134778
Slope:    29.863318947981266

Gradient Descent
Intercept: 0.14239262503773842
Slope:    2.654209102713116


4. Plot MSE against iteration. What happens to the loss as gradient descent runs? What would suggest that the algorithm has converged?
- The plot shows that MSE steadily decreases as gradient descent runs. It means as we update our coefficients through each iteration, it reduces prediction error MSE. However, as the curve is still declining and has not leveled off, the algorithm does not appear to have converged. Convergence happens when MSE almost becomes constant and there is little change between iterations.

In [65]:
results = []

for iteration, values in history.items():
    results.append({
        "iteration": iteration,
        "mse": values["residual"]
    })

results = pd.DataFrame(results)

fig = px.line(
    results,
    x="iteration",
    y="mse",
    title="MSE Against Gradient Descent Iteration",
    labels={
        "iteration": "Iteration",
        "mse": "Mean Squared Error"
    }
)

fig.show()

5. What might happen if the learning rate is much too large?
- If the learning rate is too large, gradient descent may decrease too fast and pass by the minimum. Instead of steadily decreasing, the MSE may go up and down on the curve and causes the algorith to fail to converge.

## Part 5: From Simple to Multiple Regression

1. Why might this model predict bike demand better than the simple regression model?
- I tink this model predict bike demand better because humidity, wind speed, and rainfall may also related to bike demand. For example, people may be less likely to ride bikes when it is raining or very humid, even if the temperature is comfortable. Including these variables also help the model distinguis between hours with similar temperatures. For example, two hours could be both 20C, but one is raining while the other is dry.

2. Inperpret b1, the coefficient for temperature, in this multiple regression model. Be precise about what is being held fixed.
- Holding humidity, wind speed, and rainfall constant; a 1C increase in temperature is associated with a $b_1$ change in predicted number of bike rentals in during an hour

3. If we keep adding predictions, will prediction on new observations necessarily improve?
- I don't think so. Some predictors may be irrelevant with regards to the prediction of bike rentals and by including them, we cause the model to fit randome patterns in the training data. This can lead to overfitting which will not produce good prediction for new observations. 

I did use ChatGPT to help me fix the bug in my code, especially the recording MSE in Part 4: Gradient Descent. 